In [127]:
!pip install autogluon


In [128]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [129]:
from autogluon.tabular import TabularPredictor


# accelera

In [130]:
import os
import pickle

import torch
from PIL import Image


def check_path_exists(folder_path, filename):
    path = os.path.join(folder_path, filename)
    if not os.path.exists(path):
        raise ValueError(f"{path} does not exist")
    return True


def get_sub_folders_names(folder_path):
    sub_folders_name = [
        class_name
        for class_name in os.listdir(folder_path)
        if os.path.isdir(os.path.join(folder_path, class_name))
    ]
    return sub_folders_name


def save_pickle(folder_path, obj, filename):
    filepath = os.path.join(folder_path, filename)
    with open(filepath, "wb") as f:
        pickle.dump(obj, f)


def load_pickle(folder_path, filename):
    filepath = os.path.join(folder_path, filename)
    with open(filepath, "rb") as f:
        obj = pickle.load(f)
    return obj


def lower_data(df):
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].apply(lambda x: x.lower() if isinstance(x, str) else x)


def drop_columns(X, col_drop):
    col_drop = list(col_drop.keys())
    if col_drop:
        X.drop(columns=col_drop, inplace=True, errors="ignore")


def is_valid_image(image_path):
    valid_extension = (".jpg", ".png", ".jpeg")

    if not os.path.isfile(image_path):
        return False
    if image_path.endswith(valid_extension):
        try:
            with Image.open(image_path) as img:
                img.verify()
            return True
        except Exception:
            return False
    return False


def collect_function(batch):
    images = [item[0] for item in batch]
    labels = [item[1] for item in batch]
    images_stack = torch.stack(images, dim=0)
    if labels[0] is None:
        return images_stack
    labels = torch.tensor(labels, dtype=torch.long)
    return images_stack, labels


def collect_function_segmentation(batch):
    images = [item[0] for item in batch]
    masks = [item[1] for item in batch]
    images_stack = torch.stack(images, dim=0)
    if masks[0] is None:
        return images_stack
    masks_stack = torch.stack([mask for mask in masks], dim=0)
    return images_stack, masks_stack


In [131]:
import pandas as pd

class PreprocessingBase:
    def __init__(self, folder_path):
        self.folder_path = folder_path
        if folder_path is None:
            raise ValueError("folder_path cannot be None")

    def common_preprocessing(self):
        raise NotImplementedError("Must implement common_preprocessing method.")


class TabularPreprocessingBase(PreprocessingBase):
    def __init__(self, df, folder_path=None):
        self.df = df
        if df is None:
            raise ValueError("Dataframe cannot be None")
        if not isinstance(df, pd.DataFrame):
            raise ValueError("df must be a pandas DataFrame")
        super().__init__(folder_path)


In [132]:
import io
import os

from sklearn.model_selection import train_test_split



class TrainingTabularPreprocessingBase(TabularPreprocessingBase):
    def __init__(self, df, target_col, val_size, random_state, folder_path=None):
        super().__init__(df, folder_path)
        self.target_col = target_col
        self.val_size = val_size
        self.random_state = random_state
        self.report_data = {}

        if self.target_col not in self.df.columns:
            raise ValueError("target_col must be one of the dataframe columns")

        if (not (isinstance(self.val_size, (int, float)))) or (
            not (0 <= self.val_size <= 0.5)
        ):
            raise ValueError(
                "test size is invalid it must be less than or equal 0.5"
            )

        if self.random_state is not None and not (
            isinstance(self.random_state, int)
        ):
            raise ValueError("random state is invalid it must be integer or None")

        self.target_type = self.df[self.target_col].dtype
        os.makedirs(self.folder_path, exist_ok=True)

    def data_overview(self):
        data_head = self.df.head()
        lower_data(self.df)
        lower_data_head = self.df.head()
        io_buffer = io.StringIO()
        self.df.info(buf=io_buffer)
        data_info = io_buffer.getvalue()
        numerical_df = self.df.select_dtypes(include="number")
        categorical_df = self.df.select_dtypes(include="object")
        numerical_describe, categorical_describe = None, None
        if not numerical_df.empty:
            numerical_describe = numerical_df.describe()
        if not categorical_df.empty:
            categorical_describe = categorical_df.describe()
        missing_values = self.df.isnull().sum()
        duplicates_sum = self.df.duplicated().sum()
        duplicates_percentage = self.df.duplicated().mean() * 100
        self.report_data["data_overview"] = {
            "data_head": data_head,
            "lower_data_head": lower_data_head,
            "info": data_info,
            "numerical_describe": numerical_describe,
            "categorical_describe": categorical_describe,
            "missing_values": missing_values,
            "duplicates_sum": duplicates_sum,
            "duplicates_percentage": duplicates_percentage,
            "shape": self.df.shape,
        }

    def drop_duplicates(self):
        self.df.drop_duplicates(inplace=True)
        self.report_data["drop_duplicates"] = {
            "shape": self.df.shape,
            "duplicates_sum": self.df.duplicated().sum(),
            "duplicates_percentage": self.df.duplicated().mean() * 100,
        }

    def split_data(self):
        X, y = self.df.drop(columns=[self.target_col]), self.df[self.target_col]
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=self.val_size, random_state=self.random_state
        )
        self.report_data["split"] = {
            "val_size": self.val_size,
            "X_train_shape": X_train.shape,
            "X_val_shape": X_val.shape,
            "y_train_shape": y_train.shape,
            "y_val_shape": y_val.shape,
        }
        return X_train, X_val, y_train, y_val


In [133]:
import pandas as pd

from sklearn.base import BaseEstimator
from sklearn.base import TransformerMixin


class CustomTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        raise NotImplementedError("CustomTransformer.fit is not implemented yet.")

    def transform(self, X):
        raise NotImplementedError(
            "CustomTransformer.transform is not implemented yet."
        )


class FrequencyEncoderTransform(CustomTransformer):
    def __init__(self):
        self.mapping_ = {}
        self.cols_ = None

    def fit(self, X, y=None):
        self.mapping_ = {}
        self.cols_ = None
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        self.cols_ = X.columns
        for col in self.cols_:
            self.mapping_[col] = X[col].value_counts(normalize=True).to_dict()
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self.cols_)

        X_copy = X.copy()
        for col in self.cols_:
            X_copy[col] = X_copy[col].map(self.mapping_[col]).fillna(0)
        return X_copy.values

    def get_feature_names_out(self, input_features=None):
        return input_features

import pandas as pd



class IQRTransform(CustomTransformer):
    def __init__(self, info, cols):
        self.info = info
        self.cols = cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = X.copy()
        for i, col in enumerate(self.cols):
            if self.info[col]["col_type"] in ["numerical", "continuous"]:
                lower, upper = self.info[col]["outliers_info"]
                if lower is not None and upper is not None:
                    if isinstance(X_copy, pd.DataFrame):
                        X_copy[col] = X_copy[col].clip(lower, upper)
                    else:
                        X_copy[:, i] = X_copy[:, i].clip(lower, upper)
        return X_copy

    def get_feature_names_out(self, input_features=None):
        return input_features


In [134]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler






class ClassicalTrainingPreprocessing(TrainingTabularPreprocessingBase):
    def __init__(
        self,
        df,
        target_col: str,
        problem_type="classification",
        folder_path=None,
        val_size=0.2,
        random_state=42,
        cardinality_threshold=8,
        max_unique_ordinal=10,
        missing_threshold=0.5,
        unique_threshold=0.9,
    ):
        super().__init__(df, target_col, val_size, random_state, folder_path)
        self.problem_type = problem_type
        self.cardinality_threshold = cardinality_threshold
        self.max_unique_ordinal = max_unique_ordinal
        self.missing_threshold = missing_threshold
        self.unique_threshold = unique_threshold
        if self.problem_type is None:
            raise ValueError("problem_type cannot be None")
        self.problem_type = problem_type.lower()
        if self.problem_type not in ["classification", "regression"]:
            raise ValueError(
                "problem_type must be either 'classification' or 'regression'"
            )
        if self.problem_type == "classification" and not (
            np.issubdtype(self.target_type, np.integer)
            or self.target_type == "object"
            or self.target_type == "bool"
        ):
            raise ValueError(
                "Target must be integer or object fro classification problem"
            )
        if self.problem_type == "regression" and (
            not np.issubdtype(self.target_type, np.number)
            or self.target_type == "bool"
            or self.df[self.target_col].nunique() == 2
        ):
            raise ValueError("Target must be numeric for regression problem")

        if (
            not isinstance(self.cardinality_threshold, int)
            or self.cardinality_threshold < 0
        ):
            raise ValueError("cardinality_threshold must be positive integer")

        if (
            not isinstance(self.max_unique_ordinal, int)
            or self.max_unique_ordinal < 0
        ):
            raise ValueError("max_unique_ordinal must be positive integer")
        if not isinstance(self.missing_threshold, float) or not (
            0 <= self.missing_threshold <= 1
        ):
            raise ValueError("missing_threshold must be float between 0 and 1")
        if not isinstance(self.unique_threshold, float) or not (
            0 <= self.unique_threshold <= 1
        ):
            raise ValueError("unique_threshold must be float between 0 and 1")

        save_pickle(self.folder_path, self.df.columns.tolist(), "data_columns.pkl")

    def is_drop_column(self, info, col):
        if info[col].get("is_constant", False):
            return True, "The column is constant"
        elif info[col].get("p_unique", 0) > self.unique_threshold and (
            info[col].get("dtype") == "object"
            or np.issubdtype(info[col].get("dtype"), np.integer)
        ):
            return (
                True,
                f"It is above unique_threshold {self.unique_threshold}",
            )
        elif info[col].get("p_missing", 0) > self.missing_threshold:
            return (
                True,
                f"Missing above missing_threshold {self.missing_threshold}",
            )
        return False, None

    def outliers_info(self, info, col):
        Q1 = info[col]["Q1"]
        Q3 = info[col]["Q3"]
        min_value = info[col]["min"]
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        lower = max(lower, min_value)
        upper = Q3 + 1.5 * IQR
        return (lower, upper)

    def check_binary(self, col, info):
        if info[col].get("n_unique", 0) == 2 or info[col].get("dtype") == "bool":
            return True

    def get_data_info(self, X_train, y_train):
        col_drop = {}
        info = {}
        df_new = X_train.copy()
        df_new[self.target_col] = y_train
        for col in list(df_new.columns):
            info[col] = {
                "dtype": df_new[col].dtype,
                "n_unique": df_new[col].nunique(),
                "p_unique": df_new[col].nunique() / df_new.shape[0],
                "missing": df_new[col].isna().sum(),
                "p_missing": df_new[col].isna().sum() / df_new.shape[0],
                "is_constant": df_new[col].nunique() == 1,
            }

            if np.issubdtype(df_new[col].dtype, np.number):
                info[col]["skew"] = df_new[col].skew()
                info[col]["variance"] = df_new[col].var()
                info[col]["Q1"] = df_new[col].quantile(0.25)
                info[col]["Q3"] = df_new[col].quantile(0.75)
                info[col]["min"] = df_new[col].min()
                info[col]["max"] = df_new[col].max()
                info[col]["median"] = df_new[col].median()
                info[col]["mean"] = df_new[col].mean()
                info[col]["outliers_info"] = self.outliers_info(info, col)
            mode = df_new[col].mode()
            if not mode.empty:
                info[col]["mode"] = mode[0]
            else:
                info[col]["mode"] = None

            if col != self.target_col:
                is_drop, reason = self.is_drop_column(info, col)
                if is_drop:
                    col_drop[col] = reason

        return info, col_drop

    def drop_col(self, X_train, X_val, col_drop):
        drop_columns(X_train, col_drop)
        drop_columns(X_val, col_drop)
        self.report_data["drop_columns"] = {
            "col_drop": col_drop,
            "X_trian_head": X_train.head(),
            "X_val_head": X_val.head(),
        }
        save_pickle(self.folder_path, col_drop, "col_drop.pkl")

    def detect_column_types(self, X_train, info):
        binary_cols = []
        numerical_cols = []
        one_hot_cols = []
        frequency_cols = []
        ordinal_cols = []
        others = []
        self.report_data["preprocessing"] = []
        for col in X_train.columns:
            if self.check_binary(col, info):
                info[col]["col_type"] = "binary"
                info[col]["preprossing_steps"] = [
                    "Fill missing with most frequent",
                    "Ordinal encoding",
                ]
                binary_cols.append(col)
            elif np.issubdtype(info[col]["dtype"], np.integer):
                if info[col]["n_unique"] <= self.max_unique_ordinal:
                    info[col]["col_type"] = "ordinal"
                    info[col]["preprossing_steps"] = [
                        "Fill missing with most frequent",
                        "Robust scaling",
                    ]
                    ordinal_cols.append(col)
                else:
                    info[col]["col_type"] = "numerical"
                    info[col]["preprossing_steps"] = [
                        "Fill missing with median",
                        "IQR transform",
                        "Standard scaling",
                    ]
                    numerical_cols.append(col)

            elif np.issubdtype(info[col]["dtype"], np.floating):
                info[col]["col_type"] = "continuous"
                info[col]["preprossing_steps"] = [
                    "Fill missing with median",
                    "IQR transform",
                    "Standard scaling",
                ]
                numerical_cols.append(col)

            elif info[col]["dtype"] == "object":
                if info[col]["n_unique"] <= self.cardinality_threshold:
                    info[col]["col_type"] = "low level cardinality"
                    info[col]["preprossing_steps"] = [
                        "Fill missing with most frequent",
                        "One hot encoding",
                    ]
                    one_hot_cols.append(col)
                else:
                    info[col]["col_type"] = "high level cardinality"
                    info[col]["preprossing_steps"] = [
                        "Fill missing with most frequent",
                        "Frequency encoding",
                    ]
                    frequency_cols.append(col)

            else:
                info[col]["col_type"] = "other"
                info[col]["preprossing_steps"] = (
                    f"Drop column because its type {info[col]['dtype']} "
                    f"is not supported"
                )
                others.append(col)
            self.report_data["preprocessing"].append(
                {
                    "col_name": col,
                    "col_type": info[col]["col_type"],
                    "col_preprocessing": info[col]["preprossing_steps"],
                }
            )
        return (
            binary_cols,
            numerical_cols,
            one_hot_cols,
            frequency_cols,
            ordinal_cols,
            others,
        )



    def features_preprocessing(
        self,
        X_train,
        X_val,
        info,
        binary_cols,
        numerical_cols,
        one_hot_cols,
        frequency_cols,
        ordinal_cols,
    ):
        numerical_pipeline = Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("iqr_transformer", IQRTransform(info, numerical_cols)),
                ("scaler", StandardScaler()),
            ]
        )
        one_hot_pipeline = Pipeline(
            [
                ("imputer", SimpleImputer(strategy="most_frequent")),
                (
                    "one_hot_encoder",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                        drop="first",
                    ),
                ),
            ]
        )
        frequency_pipeline = Pipeline(
            [
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("frequency_encoder", FrequencyEncoderTransform()),
            ]
        )
        binary_pipeline = Pipeline(
            [
                ("imputer", SimpleImputer(strategy="most_frequent")),
                (
                    "ordinal",
                    OrdinalEncoder(
                        handle_unknown="use_encoded_value", unknown_value=-1
                    ),
                ),
            ]
        )
        ordinal_pipeline = Pipeline(
            [
                ("imputer", SimpleImputer(strategy="most_frequent")),
               ("scaler", RobustScaler()),
            ]
        )
        preprocessor = ColumnTransformer(
            transformers=[
                ("onehot", one_hot_pipeline, one_hot_cols),
                ("numerical", numerical_pipeline, numerical_cols),
                ("binary", binary_pipeline, binary_cols),
                ("frequency", frequency_pipeline, frequency_cols),
                ("ordinal", ordinal_pipeline, ordinal_cols),
            ],
            remainder="drop",
        )
        X_train_processed = preprocessor.fit_transform(X_train)
        X_val_processed = preprocessor.transform(X_val)
        save_pickle(self.folder_path, preprocessor, "training_preprocessor.pkl")
        farures_names = [
            x.split("__")[-1] for x in preprocessor.get_feature_names_out()
        ]
        save_pickle(self.folder_path, farures_names, "feature_names.pkl")
        self.report_data["after_preprocessing"] = {
            "X_train_processed": pd.DataFrame(
                X_train_processed, columns=farures_names
            ).head(),
            "X_val_processed": pd.DataFrame(
                X_val_processed, columns=farures_names
            ).head(),
        }

        return X_train_processed, X_val_processed

    def target_preprocessing(self, y_train, y_val, info):
        target_dict = {
            "col_name": self.target_col,
            "problem_type": self.problem_type,
        }
        if self.problem_type == "classification":
            y_train = y_train.fillna(info[self.target_col]["mode"])
            y_val = y_val.fillna(info[self.target_col]["mode"])
            label_encoder = LabelEncoder()
            y_train = label_encoder.fit_transform(y_train)
            y_val = label_encoder.transform(y_val)
            save_pickle(self.folder_path, label_encoder, "target_preprocessor.pkl")
            target_dict["mode"] = info[self.target_col]["mode"]
            self.report_data["preprocessing"].append(
                {
                    "col_name": self.target_col,
                    "col_type": self.problem_type,
                    "col_preprocessing": [
                        "Fill missing with most frequent",
                        "Label encoding",
                    ],
                }
            )
        elif self.problem_type == "regression":
            y_train = y_train.fillna(info[self.target_col]["median"])
            y_val = y_val.fillna(info[self.target_col]["median"])
            target_dict["median"] = info[self.target_col]["median"]
            stander_scaler = StandardScaler()
            y_train = stander_scaler.fit_transform(
                y_train.values.reshape(-1, 1)
            ).ravel()
            y_val = stander_scaler.transform(y_val.values.reshape(-1, 1)).ravel()
            save_pickle(self.folder_path, stander_scaler, "target_preprocessor.pkl")
            self.report_data["preprocessing"].append(
                {
                    "col_name": self.target_col,
                    "col_type": self.problem_type,
                    "col_preprocessing": [
                        "Fill missing with median",
                        "Standard scaling",
                    ],
                }
            )
        self.report_data["after_preprocessing"]["y_train_processed"] = pd.DataFrame(
            y_train, columns=[self.target_col]
        ).head()
        self.report_data["after_preprocessing"]["y_val_processed"] = pd.DataFrame(
            y_val, columns=[self.target_col]
        ).head()
        save_pickle(self.folder_path, target_dict, "target_info.pkl")
        return y_train, y_val

    def common_preprocessing(self):
        self.data_overview()
        self.drop_duplicates()
        X_train, X_val, y_train, y_val = self.split_data()
        info, col_drop = self.get_data_info(X_train, y_train)
        self.drop_col(X_train, X_val, col_drop)
        (
            binary_cols,
            numerical_cols,
            one_hot_cols,
            frequency_cols,
            ordinal_cols,
            _,
        ) = self.detect_column_types(X_train, info)
        X_train, X_val = self.features_preprocessing(
            X_train,
            X_val,
            info,
            binary_cols,
            numerical_cols,
            one_hot_cols,
            frequency_cols,
            ordinal_cols,
        )
        y_train, y_val = self.target_preprocessing(y_train, y_val, info)
        return X_train, y_train, X_val, y_val


In [139]:
from sklearn.metrics import r2_score,mean_squared_error
def withAutoGloun(path,label):
    df = pd.read_csv(path)
    training_df,testing_df=train_test_split(df,test_size=0.2,random_state=42)
    predictor = TabularPredictor(label=label).fit(training_df)
    predictions = predictor.predict(testing_df.drop(columns=[label]))
    print(classification_report(testing_df[label],predictions))

def withputAutoGloun(X_train,X_val,y_train,y_val,label):
    columns = [f"feature_{i}" for i in range(X_train.shape[1])]
    X_train_df = pd.DataFrame(X_train, columns=columns)
    X_val_df = pd.DataFrame(X_val, columns=columns)
    X_train_df[label] = y_train
    X_val_df[label] = y_val
    predictor = TabularPredictor(label=label).fit(train_data=X_train_df,feature_generator=None)
    predictions = predictor.predict(X_val_df.drop(columns=[label]))
    print(classification_report(y_val,predictions))

In [140]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import mean_squared_error
from sklearn.svm import SVC


print(
    "----------------------------student_exam_performance_dataset-----------------------"
)

student_exam = pd.read_csv("/content/drive/MyDrive/Accelera datasets/student_exam_performance_dataset.csv")
training_preprocessor = ClassicalTrainingPreprocessing(
    student_exam, "pass_fail", "Classification", "./student_exam"
)
X_train, y_train, X_val, y_val = training_preprocessor.common_preprocessing()
print("-" * 10)
print("with")
print("-" * 10)
print("-" * 10)
withAutoGloun("/content/drive/MyDrive/Accelera datasets/student_exam_performance_dataset.csv","pass_fail")
print("without")
print("-" * 10)
withputAutoGloun(X_train,X_val,y_train,y_val,"pass_fail")
print("----------------------------spine Dataset-----------------------")
spine = pd.read_csv("/content/drive/MyDrive/Accelera datasets/Dataset_spine.csv")
training_preprocessor = ClassicalTrainingPreprocessing(
    spine, "Class_att", "classification", "./Dataset_spine"
)
X_train, y_train, X_val, y_val = training_preprocessor.common_preprocessing()
print("-" * 10)
print("with")
print("-" * 10)
withAutoGloun("/content/drive/MyDrive/Accelera datasets/Dataset_spine.csv","Class_att")
print("without")
withputAutoGloun(X_train,X_val,y_train,y_val,"Class_att")
print("----------------------------diabetes Dataset-----------------------")
diab = pd.read_csv("/content/drive/MyDrive/Accelera datasets/diabetes_dataset.csv")
training_preprocessor = ClassicalTrainingPreprocessing(
    diab, "diagnosed_diabetes", "classification", "./diabetes_dataset"
)
X_train, y_train, X_val, y_val = training_preprocessor.common_preprocessing()
print("-" * 10)
print("with")
print("-" * 10)
withAutoGloun("/content/drive/MyDrive/Accelera datasets/diabetes_dataset.csv","diagnosed_diabetes")
print("without")
withputAutoGloun(X_train,X_val,y_train,y_val,"diagnosed_diabetes")
print("----------------------------student Dataset-----------------------")
student = pd.read_csv("/content/drive/MyDrive/Accelera datasets/student_placement_synthetic.csv")
training_preprocessor = ClassicalTrainingPreprocessing(
    student, "placement_status", "classification", "./student_report"
)
X_train, y_train, X_val, y_val = training_preprocessor.common_preprocessing()
print("-" * 10)
print("with")
print("-" * 10)
withAutoGloun("/content/drive/MyDrive/Accelera datasets/student_placement_synthetic.csv","placement_status")
print("without")
withputAutoGloun(X_train,X_val,y_train,y_val,"placement_status")
print("----------------------------Titanic Dataset-----------------------")
titanic_df =pd.read_csv("/content/drive/MyDrive/Accelera datasets/Titanic-Dataset.csv")
training_preprocessor = ClassicalTrainingPreprocessing(
    titanic_df, "Survived", "classification", "./titanic_preprocessing"
)
X_train, y_train, X_val, y_val = training_preprocessor.common_preprocessing()
print("-" * 10)
print("with")
print("-" * 10)
withAutoGloun("/content/drive/MyDrive/Accelera datasets/Titanic-Dataset.csv","Survived")
print("without")
withputAutoGloun(X_train,X_val,y_train,y_val,"Survived")


print("----------------------------Purchase Dataset-----------------------")

purchase_df = pd.read_csv("/content/drive/MyDrive/Accelera datasets/customer_purchase_data.csv")
training_preprocessor = ClassicalTrainingPreprocessing(
    purchase_df, "PurchaseStatus", "classification", "./PurchaseStatus"
)
X_train, y_train, X_val, y_val = training_preprocessor.common_preprocessing()
print("-" * 10)
print("with")
print("-" * 10)
withAutoGloun("/content/drive/MyDrive/Accelera datasets/customer_purchase_data.csv","PurchaseStatus")
print("without")
withputAutoGloun(X_train,X_val,y_train,y_val,"PurchaseStatus")



----------------------------student_exam_performance_dataset-----------------------


No path specified. Models will be saved in: "AutogluonModels/ag-20260411_171326"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Mon Feb  2 12:27:57 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       9.62 GB / 12.67 GB (75.9%)
Disk Space Avail:   66.82 GB / 107.72 GB (62.0%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new Tabular Foundation Models (TFMs) meta-learned on https://tabarena.a

----------
with
----------
----------


Beginning AutoGluon training ...
AutoGluon will save models to "/content/AutogluonModels/ag-20260411_171326"
Train Data Rows:    8000
Train Data Columns: 16
Label Column:       pass_fail
AutoGluon infers your prediction problem is: 'binary' (because only two unique label-values observed).
	2 unique label values:  ['Pass', 'Fail']
	If 'binary' is not the correct problem_type, please manually specify the problem_type parameter during Predictor init (You may specify problem_type as one of: ['binary', 'multiclass', 'regression', 'quantile'])
Problem Type:       binary
Preprocessing data ...
Selected class <--> label mapping:  class 1 = Pass, class 0 = Fail
	Note: For your binary classification, AutoGluon arbitrarily selected which label-value represents positive (Pass) vs negative (Fail) class.
	To explicitly set the positive_class, either rename classes to 1 and 0, or specify positive_class in Predictor init.
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatur

              precision    recall  f1-score   support

        Fail       0.87      0.86      0.87      1070
        Pass       0.84      0.85      0.85       930

    accuracy                           0.86      2000
   macro avg       0.86      0.86      0.86      2000
weighted avg       0.86      0.86      0.86      2000

without
----------


...

              precision    recall  f1-score   support

           0       0.80      0.90      0.85       105
           1       0.83      0.68      0.75        74

    accuracy                           0.81       179
   macro avg       0.82      0.79      0.80       179
weighted avg       0.81      0.81      0.81       179

----------------------------Purchase Dataset-----------------------
----------
with
----------


Beginning AutoGluon training ...
AutoGluon will save models to "/content/AutogluonModels/ag-20260411_175308"
Train Data Rows:    1200
Train Data Columns: 8
Label Column:       PurchaseStatus
AutoGluon infers your prediction problem is: 'binary' (because only two unique label-values observed).
	2 unique label values:  [np.int64(0), np.int64(1)]
	If 'binary' is not the correct problem_type, please manually specify the problem_type parameter during Predictor init (You may specify problem_type as one of: ['binary', 'multiclass', 'regression', 'quantile'])
Problem Type:       binary
Preprocessing data ...
Selected class <--> label mapping:  class 1 = 1, class 0 = 0
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    10061.48 MB
	Train Data (Original)  Memory Usage: 0.07 MB (0.0% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special d

              precision    recall  f1-score   support

           0       0.94      0.98      0.96       172
           1       0.97      0.92      0.94       128

    accuracy                           0.95       300
   macro avg       0.96      0.95      0.95       300
weighted avg       0.95      0.95      0.95       300

without


Beginning AutoGluon training ...
AutoGluon will save models to "/content/AutogluonModels/ag-20260411_175333"
Train Data Rows:    1110
Train Data Columns: 8
Label Column:       PurchaseStatus
AutoGluon infers your prediction problem is: 'binary' (because only two unique label-values observed).
	2 unique label values:  [np.int64(0), np.int64(1)]
	If 'binary' is not the correct problem_type, please manually specify the problem_type parameter during Predictor init (You may specify problem_type as one of: ['binary', 'multiclass', 'regression', 'quantile'])
Problem Type:       binary
Preprocessing data ...
Selected class <--> label mapping:  class 1 = 1, class 0 = 0
Using Feature Generators to preprocess the data ...
Fitting IdentityFeatureGenerator...
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
Data preprocessing and feature engineering runtime = 0.02s ...
AutoGluon will gauge predictive performance

              precision    recall  f1-score   support

           0       0.92      0.95      0.93       149
           1       0.94      0.91      0.92       129

    accuracy                           0.93       278
   macro avg       0.93      0.93      0.93       278
weighted avg       0.93      0.93      0.93       278

